In [ ]:
"""

Final prediction pipeline per horizon:
  a. Blend L1 + L2 base predictions (α tuned on validation).
  b. Apply spike policy (soft blend / gated uplift / hard gate — kind and
     parameters tuned on validation via _spike_policy_score).
  c. Apply dip policy (same three kinds, tuned independently).
  d. Blend model prediction with lag-naive baseline (α tuned on validation).
  e. Isotonic regression calibration fitted on validation set.

Additionally:
  - Per-horizon target-time features (8 feats: hour/dow sin-cos + regime flags)
  - Cross-features (4 feats: doy cycle, SA-NSW spread, region spike count)
  - Exponential recency weights (newest row ~4.5x oldest)



"""

In [ ]:

PRICE_TRANSFORM_SCALE = 100.0



def compute_cross_feats(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with 4 auxiliary cross-feature columns.

    Columns
    -------
    doy_sin            : 365.25-day annual sinusoidal cycle
    doy_cos            : 365.25-day annual cosine cycle
    sa_spread_live     : arcsinh((SA - NSW) / scale) — SA-NSW interconnector pressure
    region_spike_score : count of QLD/VIC/SA regions with price > 150 (0–3)
    """
    n   = len(df)
    doy = df.index.day_of_year.astype(np.float32)

    doy_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    doy_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    if "sa_price" in df.columns and "nsw_price" in df.columns:
        sa  = df["sa_price"].fillna(0.0).values.astype(np.float32)
        nsw = df["nsw_price"].fillna(0.0).values.astype(np.float32)
        sa_spread = np.arcsinh((sa - nsw) / PRICE_TRANSFORM_SCALE).clip(-10, 10).astype(np.float32)
    elif "nsw_price_vs_sa_price_spread" in df.columns:
        sa_spread = np.arcsinh(
            df["nsw_price_vs_sa_price_spread"].fillna(0.0).values / PRICE_TRANSFORM_SCALE
        ).clip(-10, 10).astype(np.float32)
    else:
        sa_spread = np.zeros(n, dtype=np.float32)

    score = np.zeros(n, dtype=np.float32)
    for reg_col in ["qld_price", "vic_price", "sa_price"]:
        if reg_col in df.columns:
            score += (df[reg_col].fillna(0.0).values > 150).astype(np.float32)

    return pd.DataFrame({
        "doy_sin":            doy_sin,
        "doy_cos":            doy_cos,
        "sa_spread_live":     sa_spread,
        "region_spike_score": score,
    }, index=df.index)



# Price col (clip threshold source) + naive baseline lags (blend baseline).
_price_lag_cols = [
    f"{os.environ.get("TARGET").lower()}_price",
    f"{os.environ.get("TARGET").lower()}_price_lag_336",
    f"{os.environ.get("TARGET").lower()}_price_lag_48",
]
_price_lag_cols = [c for c in _price_lag_cols if c in df.columns]

auxiliary = pd.concat([cross, df[_price_lag_cols]], axis=1)
    # # train_aux = auxiliary.loc[train_df.index]
    # # valid_aux = auxiliary.loc[validate_df.index]


 # Cross-features pre-computed in 2_Features_build/9_auxiliary_features.ipynb.
    # _CROSS_COLS = ["doy_sin", "doy_cos", "sa_spread_live", "region_spike_score"]
    # cross_tr = train_aux[_CROSS_COLS].values.astype(np.float32)
    # cross_va = valid_aux[_CROSS_COLS].values.astype(np.float32)


# aux_vars = train_aux[f"{os.environ["TARGET"].lower()}_price"].values if f"{os.environ["TARGET"].lower()}_price" in train_aux.columns else (
#     train_tgt[[c for c in train_tgt.columns if c.startswith("h") and c[1:].isdigit()]].values.ravel()
# )

# aux_vars = aux_vars[np.isfinite(aux_vars)]

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# # Price inverse transform — used in 3_evaluate to convert predictions back to $/MWh
# def _from_asinh(y: np.ndarray) -> np.ndarray:
#     return np.sinh(y) * float(PRICE_TRANSFORM_SCALE)

In [ ]:
# import numpy as np

# # Expanded validation-tuning search grids (more compute → broader search).
# _SPIKE_THR_GRID     = np.array([0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50], dtype=np.float32)
# _SPIKE_POW_GRID     = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0], dtype=np.float32)
# _SPIKE_WMAX_GRID    = np.array([0.40, 0.55, 0.70, 0.85, 1.00], dtype=np.float32)
# _SPIKE_GATE_W_GRID  = np.array([0.40, 0.60, 0.80, 1.00, 1.20], dtype=np.float32)

# _DIP_THR_GRID       = np.array([0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30], dtype=np.float32)
# _DIP_POW_GRID       = np.array([0.5, 1.0, 1.5, 2.0, 3.0], dtype=np.float32)
# _DIP_WMAX_GRID      = np.array([0.30, 0.50, 0.70, 0.90], dtype=np.float32)
# _DIP_GATE_W_GRID    = np.array([0.40, 0.60, 0.80, 1.00], dtype=np.float32)


In [ ]:
# # ── Evaluation ─────────────────────────────────────────────────────────────────

# def evaluate_seq2seq(
#     model: dict,
#     df: pd.DataFrame,
#     past_cols: list[str],
#     scaler: None,
# ) -> dict:
#     # Restore the gap/horizon the model was trained with so all internals use
#     # the correct values (number of horizons, target column names, etc.).
#     gap     = model.get("gap",     FORECAST_GAP)
#     horizon = model.get("horizon", FORECAST_HORIZON)
#     del scaler

#     df = _add_extra_lags(df)
#     _, _, test_df = _temporal_split(df)
#     X_test   = test_df[past_cols]
#     cross_te = _compute_cross_feats(test_df)

#     base_models     = model.get("base_models",     model.get("models", []))
#     base_l2_models  = model.get("base_l2_models",  [None] * horizon)
#     blend_l2_alphas = model.get("blend_l2_alphas", np.zeros(horizon, dtype=np.float32))
#     spike_clfs      = model.get("spike_clfs",      [None] * horizon)
#     spike_regs   = model.get("spike_regs",   [None] * horizon)
#     spike_qregs  = model.get("spike_qregs",  [None] * horizon)
#     dip_clfs     = model.get("dip_clfs",     [None] * horizon)
#     dip_regs     = model.get("dip_regs",     [None] * horizon)
#     blend_alphas = model.get("blend_alphas", np.ones(horizon, dtype=np.float32))
#     spike_kind   = model.get("spike_kind",   np.ones(horizon, dtype=np.int8))
#     spike_src    = model.get("spike_src",    np.zeros(horizon, dtype=np.int8))
#     spike_p1     = model.get("spike_p1",     np.full(horizon, 0.20, dtype=np.float32))
#     spike_p2     = model.get("spike_p2",     np.full(horizon, 2.0, dtype=np.float32))
#     spike_p3     = model.get("spike_p3",     np.full(horizon, 0.75, dtype=np.float32))
#     dip_kind     = model.get("dip_kind",     np.zeros(horizon, dtype=np.int8))
#     dip_p1       = model.get("dip_p1",       np.full(horizon, 0.15, dtype=np.float32))
#     dip_p2       = model.get("dip_p2",       np.full(horizon, 1.0, dtype=np.float32))
#     dip_p3       = model.get("dip_p3",       np.full(horizon, 0.60, dtype=np.float32))
#     calibrators  = model.get("calibrators",  [None] * horizon)

#     preds_m = np.empty((len(test_df), horizon), dtype=np.float32)

#     for i, h in enumerate(range(gap, gap + horizon)):
#         h_feat_te  = _compute_target_time_feats(test_df.index, h)
#         pd_feat_te = _compute_aligned_pd_feats(test_df, h)
#         X_h_te = np.concatenate([X_test.values, h_feat_te, cross_te, pd_feat_te], axis=1).astype(np.float32)

#         _l1_te = _from_asinh(base_models[i].predict(X_h_te)).astype(np.float32)
#         _l2_m  = base_l2_models[i] if (i < len(base_l2_models) and base_l2_models[i] is not None) else None
#         if _l2_m is not None:
#             _l2_te = _from_asinh(_l2_m.predict(X_h_te)).astype(np.float32)
#             _la    = float(blend_l2_alphas[i]) if i < len(blend_l2_alphas) else 0.0
#             y_base = ((1.0 - _la) * _l1_te + _la * _l2_te).astype(np.float32)
#         else:
#             y_base = _l1_te
#         if i < len(spike_clfs) and spike_clfs[i] is not None:
#             sp = spike_clfs[i].predict_proba(X_h_te)[:, 1].astype(np.float32)
#             if i < len(spike_regs) and spike_regs[i] is not None:
#                 sr = _from_asinh(spike_regs[i].predict(X_h_te)).astype(np.float32)
#             else:
#                 sr = y_base
#             if i < len(spike_qregs) and spike_qregs[i] is not None:
#                 sq = _from_asinh(spike_qregs[i].predict(X_h_te)).astype(np.float32)
#             else:
#                 sq = sr
#             kind = int(spike_kind[i]) if i < len(spike_kind) else 1
#             src  = int(spike_src[i]) if i < len(spike_src) else 0
#             p1   = float(spike_p1[i]) if i < len(spike_p1) else 0.20
#             p2   = float(spike_p2[i]) if i < len(spike_p2) else 2.0
#             p3   = float(spike_p3[i]) if i < len(spike_p3) else 0.75
#             y_model = _apply_spike_policy(y_base, sp, sr, sq, kind, p1, p2, p3, src)
#         else:
#             y_model = y_base

#         if i < len(dip_clfs) and dip_clfs[i] is not None and i < len(dip_regs) and dip_regs[i] is not None:
#             dp = dip_clfs[i].predict_proba(X_h_te)[:, 1].astype(np.float32)
#             dr = _from_asinh(dip_regs[i].predict(X_h_te)).astype(np.float32)
#             dkind = int(dip_kind[i]) if i < len(dip_kind) else 0
#             dp1   = float(dip_p1[i]) if i < len(dip_p1) else 0.15
#             dp2   = float(dip_p2[i]) if i < len(dip_p2) else 1.0
#             dp3   = float(dip_p3[i]) if i < len(dip_p3) else 0.60
#             y_model = _apply_dip_policy(y_model, dp, dr, dkind, dp1, dp2, dp3)

#         a         = float(blend_alphas[i]) if i < len(blend_alphas) else 1.0
#         naive_col = _horizon_naive_col(h)
#         if naive_col not in test_df.columns:
#             naive_col = "price_lag_48"
#         if naive_col in test_df.columns and a < 1.0:
#             naive_h = test_df[naive_col].values.astype(np.float32)
#             y_blend = a * y_model + (1.0 - a) * naive_h
#         else:
#             y_blend = y_model

#         cal = calibrators[i] if i < len(calibrators) else None
#         if cal is not None:
#             y_blend = cal.predict(y_blend.astype(np.float64)).astype(np.float32)
#         preds_m[:, i] = y_blend

#     trues_m = np.empty((len(test_df), horizon), dtype=np.float32)
#     for i, h in enumerate(range(gap, gap + horizon)):
#         trues_m[:, i] = test_df[f"target_h{h}"].values.astype(np.float32)

#     valid_rows = np.all(np.isfinite(trues_m), axis=1)
#     preds_m    = preds_m[valid_rows]
#     trues_m    = trues_m[valid_rows]
#     test_idx   = test_df.index[valid_rows]

#     def _rmse(yt: np.ndarray, yp: np.ndarray) -> float:
#         return float(np.sqrt(mean_squared_error(yt, yp)))

#     def _wmape(yt: np.ndarray, yp: np.ndarray) -> float:
#         return float(np.sum(np.abs(yt - yp)) / (np.sum(np.abs(yt)) + 1e-8) * 100)

#     def _mbe(yt: np.ndarray, yp: np.ndarray) -> float:
#         return float(np.mean(yp - yt))

#     def _median_ae(yt: np.ndarray, yp: np.ndarray) -> float:
#         return float(np.median(np.abs(yt - yp)))

#     def _build_metrics(yt: np.ndarray, yp: np.ndarray) -> dict:
#         return {
#             "mae":       float(mean_absolute_error(yt, yp)),
#             "rmse":      _rmse(yt, yp),
#             "r2":        float(r2_score(yt, yp)),
#             "mbe":       _mbe(yt, yp),
#             "median_ae": _median_ae(yt, yp),
#             "wmape":     _wmape(yt, yp),
#         }

#     yt_flat = trues_m.ravel()
#     yp_flat = preds_m.ravel()
#     model_metrics = _build_metrics(yt_flat, yp_flat)

#     # Spike-specific error analysis.
#     spike_mask = yt_flat > SPIKE_THRESHOLD
#     if spike_mask.sum() > 0:
#         model_metrics["spike_mae"]    = float(mean_absolute_error(yt_flat[spike_mask], yp_flat[spike_mask]))
#         model_metrics["nonspike_mae"] = float(mean_absolute_error(yt_flat[~spike_mask], yp_flat[~spike_mask]))
#         model_metrics["spike_pct"]    = float(spike_mask.mean() * 100)
#     dip_mask = yt_flat < _DIP_THRESHOLD
#     if dip_mask.sum() > 0:
#         model_metrics["dip_mae"] = float(mean_absolute_error(yt_flat[dip_mask], yp_flat[dip_mask]))
#         model_metrics["dip_pct"] = float(dip_mask.mean() * 100)

#     if "price_lag_48" in test_df.columns:
#         naive_base    = test_df["price_lag_48"].values[valid_rows]
#         naive_m       = np.repeat(naive_base[:, None], horizon, axis=1)
#         naive_metrics = _build_metrics(trues_m.ravel(), naive_m.ravel())
#     else:
#         naive_metrics = model_metrics

#     step_records = []
#     for i, h in enumerate(range(gap, gap + horizon)):
#         y_t = trues_m[:, i]
#         y_p = preds_m[:, i]
#         step_records.append({
#             "step":   i + 1,
#             "h":      h,
#             "lead_h": round(h * INTERVAL_MINUTES / 60, 1),
#             "mae":    round(float(mean_absolute_error(y_t, y_p)), 2),
#             "rmse":   round(_rmse(y_t, y_p), 2),
#             "r2":     round(float(r2_score(y_t, y_p)), 4),
#             "mbe":    round(_mbe(y_t, y_p), 2),
#         })

#     results_df = pd.DataFrame(index=test_idx)
#     for i, h in enumerate(range(gap, gap + horizon)):
#         results_df[f"actual_h{h}"]    = trues_m[:, i]
#         results_df[f"predicted_h{h}"] = preds_m[:, i]

#     # Feature importance from base models (spike models excluded to keep names consistent).
#     fi = np.mean([m.feature_importances_ for m in base_models], axis=0)
#     all_feat_names = list(past_cols) + _TARGET_TIME_FEAT_NAMES + _CROSS_FEAT_NAMES + _PD_ALIGNED_FEAT_NAMES
#     if len(fi) == len(all_feat_names):
#         fi_df = (
#             pd.DataFrame({"feature": all_feat_names, "importance": fi})
#             .sort_values("importance", ascending=False)
#             .reset_index(drop=True)
#         )
#     else:
#         fi_df = None  # shape guard

#     return {
#         "model":              model_metrics,
#         "naive":              naive_metrics,
#         "results_df":         results_df,
#         "steps_df":           pd.DataFrame(step_records),
#         "feature_importance": fi_df,
#     }


In [ ]:
# # ── Evaluate ───────────────────────────────────────────────────────────────────
# # Run 2_train.ipynb in the same kernel first to populate: model, df, feature_cols
# eval_output = evaluate_seq2seq(model, df, feature_cols, None)
# print(f"\n  Overall MAE: {eval_output['model']['mae']:.2f} $/MWh", flush=True)
